In [1]:
import os
import sys
import time
import threading
import queue
import numpy as np
import struct
from pynq import allocate
import cv2

# --- 1. 环境初始化 ---
WORK_DIR = '/home/xilinx/jupyter_notebooks/duxu/pynq_vqvae'
sys.path.append('/usr/lib/python3/site-packages')
sys.path.insert(0, '/home/xilinx/jupyter_notebooks/soft/DPU-PYNQ')

from pynq_dpu import DpuOverlay
import vart
import xir

In [2]:
# 加载硬件
overlay = DpuOverlay(os.path.join(WORK_DIR, 'pl_vq_31ms_260429/dpu.bit'))
vq_ip = overlay.vq_accel_1

In [3]:
# --- 2. 硬件参数与 Buffer 准备 ---
# 获取子图与 Runner
def get_dpu_subgraph(path):
    graph = xir.Graph.deserialize(path)
    return graph, graph.get_root_subgraph().toposort_child_subgraph()[1] # 这里的索引根据实际 xmodel 结构调整

enc_graph, enc_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/encoder_zcu111_700x500_old.xmodel'))
enc_runner = vart.Runner.create_runner(enc_subgraph, "run")


In [4]:
# 固定参数
enc_out_scale = 0.015625 # 2^-6
dec_in_scale  = 0.03125  # 2^-5
num_vectors = 125 * 175
dim = 64

# 分配 VQ 专用 Buffer
vq_in_z = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=0)
vq_in_cb = allocate(shape=(512, 64), dtype=np.float32, cacheable=0)
vq_out_zq = allocate(shape=(num_vectors, dim), dtype=np.int8, cacheable=0)

# 加载 Codebook
vq_in_cb[:] = np.load(os.path.join(WORK_DIR, 'codebook.npy')).astype(np.float32)
vq_in_cb.sync_to_device()

# 静态配置寄存器地址
vq_ip.register_map.in_z_1 = vq_in_z.device_address & 0xFFFFFFFF
vq_ip.register_map.in_z_2 = vq_in_z.device_address >> 32
vq_ip.register_map.in_codebook_1 = vq_in_cb.device_address & 0xFFFFFFFF
vq_ip.register_map.in_codebook_2 = vq_in_cb.device_address >> 32
vq_ip.register_map.out_z_q_1 = vq_out_zq.device_address & 0xFFFFFFFF
vq_ip.register_map.out_z_q_2 = vq_out_zq.device_address >> 32

def write_float(offset, val):
    f_hex = struct.unpack('<I', struct.pack('<f', float(val)))[0]
    vq_ip.mmio.write(offset, f_hex)

write_float(0x34, enc_out_scale)
write_float(0x3c, 1.0 / dec_in_scale)

In [5]:
# --- 3. 流水线定义 ---
PRE_DIR = './imgs_preprocessed'
data_files = sorted([f for f in os.listdir(PRE_DIR) if f.endswith('.npy')])
num_imgs = len(data_files)

dpu_res_queue = queue.Queue(maxsize=2)

def dpu_worker():
    print("🧵 [DPU Thread] Started")
    for f in data_files:
        input_data = np.load(os.path.join(PRE_DIR, f))
        
        # 核心修改1：每次循环必须分配新的 Buffer
        in_buf = [np.ascontiguousarray(input_data[np.newaxis])]
        out_buf = [np.empty((1, 125, 175, 64), dtype=np.int8, order='C')]
        
        job_id = enc_runner.execute_async(in_buf, out_buf)
        enc_runner.wait(job_id)
        
        # 核心修改2：存入队列时必须用 .copy()，否则下一帧 DPU 会重写这块内存
        dpu_res_queue.put(out_buf[0].copy())
        
    dpu_res_queue.put(None)
    print("🧵 [DPU Thread] Finished")

In [6]:
# def run_pipeline():
#     t_dpu = threading.Thread(target=dpu_worker)
#     t_dpu.start()

#     print(f"🚀 Starting Parallel Pipeline: {num_imgs} images")
#     start_time = time.time()

#     count = 0
#     while True:
#         dpu_data = dpu_res_queue.get()
#         if dpu_data is None: break
        
#         # VQ 并行段 (当 DPU 在跑下一帧时，VQ 跑当前帧)
#         vq_in_z[:] = dpu_data.reshape(num_vectors, dim)
#         vq_in_z.sync_to_device()
        
#         vq_ip.register_map.CTRL.AP_START = 1
#         while not vq_ip.register_map.CTRL.AP_DONE:
#             pass
        
#         vq_out_zq.sync_from_device()
#         # z_q_ready = vq_out_zq.copy() # 如果需要后续 Decoder 推理
        
#         print(f"  [Frame {count}] Pipeline Stage Done")
#         count += 1

#     t_dpu.join()
#     total_time = (time.time() - start_time) * 1000
#     print("\n" + "="*40)
#     print(f"Performance Report:")
#     print(f"Total Time  : {total_time:.2f} ms")
#     print(f"Average Latency : {total_time/num_imgs:.2f} ms/image")
#     print(f"Throughput  : {1000/(total_time/num_imgs):.2f} FPS")
#     print("="*40)

# if __name__ == "__main__":
#     run_pipeline()
#     del enc_runner

In [7]:
# ... 前面的初始化代码保持不变 ...

# --- 4. 初始化 Decoder 用于结果验证 ---
dec_graph, dec_subgraph = get_dpu_subgraph(os.path.join(WORK_DIR, 'xmodel/decoder_zcu111_700x500_old.xmodel'))
dec_runner = vart.Runner.create_runner(dec_subgraph, "run")
dec_out_scale = 0.007812 # 2^-7，根据你的具体 xmodel 调整

def run_pipeline():
    t_dpu = threading.Thread(target=dpu_worker)
    t_dpu.start()

    print(f"🚀 Starting Parallel Pipeline: {num_imgs} images")
    all_zq_results = [] # 用于存放结果进行验证
    start_time = time.time()

    count = 0
    while True:
        dpu_data = dpu_res_queue.get()
        if dpu_data is None: break
        
        # 核心修改3：将数据拷贝进硬件 Buffer 
        # vq_in_z 是 allocate 分配的，必须 sync_to_device
        vq_in_z[:] = dpu_data.reshape(num_vectors, dim)
        vq_in_z.sync_to_device()
        
        # 启动硬件
        vq_ip.register_map.CTRL.AP_START = 1
        while not vq_ip.register_map.CTRL.AP_DONE:
            pass
        
        # 核心修改4：从硬件取回数据
        vq_out_zq.sync_from_device()
        
        # 核心修改5：存入列表时必须用 .copy()
        # 否则 all_zq_results 里的每一项都会变成最后一帧的结果
        all_zq_results.append(vq_out_zq.copy())
        
        print(f"  [Frame {count}] Input Mean: {np.mean(dpu_data):.4f}, Output Mean: {np.mean(vq_out_zq):.4f}")
        count += 1

    t_dpu.join()
    total_time = (time.time() - start_time) * 1000
    print(f"\n✅ 流水线执行完成，耗时: {total_time:.2f} ms")

    # --- 5. 结果保存与可视化验证 ---
    print("\n🖼️ 正在保存前 3 张结果图以供验证...")
    if not os.path.exists('./results'): os.makedirs('./results')
    
    for i in range(min(3, len(all_zq_results))):
        # 1. 将 VQ 输出的 int8 转为 Decoder 需要的形状和量化值
        z_q_int8 = all_zq_results[i].reshape(1, 125, 175, 64)
        
        # 2. 运行 Decoder
        dec_in_buf = [z_q_int8]
        dec_out_buf = [np.empty((1, 500, 700, 3), dtype=np.int8, order='C')]
        job_id = dec_runner.execute_async(dec_in_buf, dec_out_buf)
        dec_runner.wait(job_id)
        
        # 3. 反归一化回 RGB 图像
        recon = dec_out_buf[0].astype(np.float32) * dec_out_scale # 转为 fp32
        recon = (recon * 0.5 + 0.5).clip(0, 1) # 反归一化 [-1,1] -> [0,1]
        recon_img = (recon[0] * 255).astype(np.uint8)
        
        # 4. 保存
        save_name = f'./results/recon_{i}.png'
        cv2.imwrite(save_name, cv2.cvtColor(recon_img, cv2.COLOR_RGB2BGR))
        print(f" ✅ 已保存结果至: {save_name}")

    print("\n" + "="*40)
    print(f"Performance: {1000/(total_time/num_imgs):.2f} FPS")
    print("="*40)

if __name__ == "__main__":
    run_pipeline()
    del enc_runner
    del dec_runner

🧵 [DPU Thread] Started
🚀 Starting Parallel Pipeline: 10 images
  [Frame 0] Input Mean: -2.5887, Output Mean: -1.3349
  [Frame 1] Input Mean: -1.4510, Output Mean: -1.3349
  [Frame 2] Input Mean: -4.5477, Output Mean: -1.3349
  [Frame 3] Input Mean: -3.9746, Output Mean: -1.3349
  [Frame 4] Input Mean: -4.7187, Output Mean: -1.3349
  [Frame 5] Input Mean: -1.5857, Output Mean: -1.3349
  [Frame 6] Input Mean: -3.5641, Output Mean: -1.3349
  [Frame 7] Input Mean: -3.0873, Output Mean: -1.3349
🧵 [DPU Thread] Finished
  [Frame 8] Input Mean: -4.8326, Output Mean: -1.3349
  [Frame 9] Input Mean: -2.6231, Output Mean: -1.3349

✅ 流水线执行完成，耗时: 431.89 ms

🖼️ 正在保存前 3 张结果图以供验证...
 ✅ 已保存结果至: ./results/recon_0.png
 ✅ 已保存结果至: ./results/recon_1.png
 ✅ 已保存结果至: ./results/recon_2.png

Performance: 23.15 FPS
